# Target encoding, step by step

Target encoding replaces a categorical value (e.g. `city = "berlin"`) with a **number
derived from the target** — typically the mean of `y` over all rows of that category.
It is popular for high-cardinality categoricals where one-hot encoding explodes.

The danger: the feature is *built from the target*, so it is easy to leak the answer
into the input. This notebook walks through every step on a dataset small enough to
check each number by hand:

1. A tiny dataset (15 rows, 3 cities)
2. Naive encoding by hand — and where the leak sits
3. Smoothing
4. Folds
5. Out-of-fold (OOF) encoding by hand, for one fold
6. The library's `oof_target_encode` — verify it matches
7. Naive vs. OOF for a single row
8. The remaining second-order channel

**How to read this notebook:** sections 1–7 are the core — what target encoding is,
where the leak sits, and how OOF encoding fixes it. If you take away only one thing:
*a row's feature value must never contain that row's own target.* Sections 8–12 are a
deep dive into a much smaller, second-order effect and can be skipped on first read —
come back to them when the core feels comfortable.

Run each cell and compare the printed tables with the text.

In [ ]:
import numpy as np
import pandas as pd

from tabular_agent.encoders import TargetEncoder, oof_target_encode
from tabular_agent.cv import make_folds

pd.set_option("display.precision", 3)

## 1. The dataset

15 rows, one categorical feature `city`, binary target `y`. The targets are
hand-picked (not random) so the category means are easy to remember:

| city | targets | mean |
|---|---|---|
| berlin | 1 1 1 0 1 | 0.8 |
| paris  | 0 0 1 0 0 | 0.2 |
| tokyo  | 1 0 1 0 1 | 0.6 |

Global mean: 8/15 ≈ 0.533.

In [ ]:
df = pd.DataFrame({
    "city": ["berlin"] * 5 + ["paris"] * 5 + ["tokyo"] * 5,
})
y = np.array([1, 1, 1, 0, 1,   # berlin -> mean 0.8
              0, 0, 1, 0, 0,   # paris  -> mean 0.2
              1, 0, 1, 0, 1])  # tokyo  -> mean 0.6

df["y"] = y
df

## 2. Naive target encoding, by hand

The simplest version: group by `city`, take the mean of `y`, map it back onto every row.

In [ ]:
category_means = df.groupby("city")["y"].mean()
category_means

In [ ]:
df_naive = df.copy()
df_naive["city_te_naive"] = df["city"].map(category_means)
df_naive

**The leak, concretely.** Look at row 0: `berlin`, `y = 1`, encoded value `0.8`.
That 0.8 is `(1+1+1+0+1)/5` — **row 0's own target is one of the five numbers in its
own feature value.** Every row partially contains its own answer.

With 5 rows per category the dilution is 1/5. With a high-cardinality feature
(categories of size 1–3), a row's feature value is *mostly* its own target — the model
can read the label straight out of the input. That is first-order leakage, the
≈0.67-AUC-on-noise effect from the walkthrough notebook.

## 3. Smoothing

Rare categories are a second problem (independent of leakage): a category seen twice
gets a mean estimated from 2 rows — pure noise. The standard fix shrinks small
categories toward the global mean:

$$\text{smoothed} = \frac{\text{mean} \cdot \text{count} \; + \; \text{global mean} \cdot s}{\text{count} + s}$$

With smoothing $s = 2$ and berlin (mean 0.8, count 5, global 0.533):
$(0.8 \cdot 5 + 0.533 \cdot 2) / (5 + 2) = 0.724$ — pulled toward the middle.
A hypothetical 1-row category with `y=1` would land at
$(1 \cdot 1 + 0.533 \cdot 2)/3 = 0.689$ instead of a fully-trusted 1.0.

By hand:

In [ ]:
s = 2.0
global_mean = df["y"].mean()
stats = df.groupby("city")["y"].agg(["mean", "count"])
stats["smoothed"] = (stats["mean"] * stats["count"] + global_mean * s) / (stats["count"] + s)
stats

The library's `TargetEncoder` implements exactly this formula. Confirm it produces
the same numbers (`fit` learns the mapping, `transform` applies it):

In [ ]:
enc = TargetEncoder(columns=["city"], smoothing=2.0).fit(df, y)
enc.mapping_["city"]

Same three values as the `smoothed` column above. Note this encoder, used alone, is
still the *naive* scheme — it was fit on all 15 rows, own targets included. Smoothing
dilutes the leak but does not remove it.

*(Default smoothing in the library is 10; we use 2 here so the numbers stay close to
the raw means and are easier to follow.)*

## 4. Folds

The fix for first-order leakage: a row's encoding must be computed **without its own
row**. Doing that per row is expensive, so it's done per fold — split the data into
folds, and encode each fold using only the *other* folds.

The library creates folds once, up front, and passes the same folds everywhere
(encoding *and* model evaluation). 3 folds of 5 rows each:

In [ ]:
folds = make_folds(y, task_type="binary", n_splits=3, seed=42)

fold_of_row = np.empty(len(df), dtype=int)
for i, (train_idx, val_idx) in enumerate(folds):
    fold_of_row[val_idx] = i

df_folds = df.copy()
df_folds["fold"] = fold_of_row
df_folds

Each row belongs to exactly one fold (its *validation* fold). `folds` is a list of
`(train_indices, val_indices)` pairs — for fold 0, the train indices are all rows of
folds 1 and 2. Stratification keeps the 0/1 balance similar across folds.

## 5. OOF encoding by hand — fold 0 only

To encode the rows of fold 0, fit the encoder **only on the rows of folds 1 and 2**
(10 rows), then apply it to fold 0's rows. First, the training rows and what they say
about each city:

In [ ]:
train_idx, val_idx = folds[0]
print("fold 0 validation rows:", val_idx)
print("fold 0 training rows:  ", train_idx)

train_df = df.iloc[train_idx]
train_stats = train_df.groupby("city")["y"].agg(["mean", "count"])
train_stats["smoothed"] = (
    train_stats["mean"] * train_stats["count"] + train_df["y"].mean() * s
) / (train_stats["count"] + s)
train_stats

These means differ from section 3: each city is now missing the 1–2 of its rows that
sit in fold 0 — precisely the rows we are about to encode. Apply the mapping to fold 0:

In [ ]:
fold0_encoded = df.iloc[val_idx].copy()
fold0_encoded["city_te_oof"] = fold0_encoded["city"].map(train_stats["smoothed"])
fold0_encoded

Pick any row here and check: its own `y` is **not** among the targets that produced
its encoded value — its row was in fold 0, and the mean came from folds 1 and 2 only.
That is the entire idea of OOF encoding. Repeating this for folds 1 and 2 fills in the
other ten rows.

## 6. The library's `oof_target_encode`

`oof_target_encode(df, y, columns, folds)` does the loop from section 5 for every fold
and returns two things:

- `oof`: a DataFrame with the OOF-encoded column — **use this for cross-validated training**
- `final`: an encoder fit on *all* rows — **use this to encode genuinely new data at
  prediction time** (safe there, because new rows have no target to leak)

In [ ]:
oof, final = oof_target_encode(df[["city"]], y, columns=["city"], folds=folds, smoothing=2.0)

overview = df_folds.copy()
overview["city_te_naive"] = df["city"].map(enc.mapping_["city"])
overview["city_te_oof"] = oof["city_te"]
overview

Two things to verify against the earlier sections:

1. The fold-0 rows match our hand computation from section 5 (the assert below).
2. The naive column has **one value per city**; the OOF column has **one value per
   (city, fold)** — same city, different number depending on which rows were excluded.

In [ ]:
manual = fold0_encoded["city_te_oof"].to_numpy()
library = oof["city_te"].to_numpy()[val_idx]
assert np.allclose(manual, library), (manual, library)
print("fold 0: manual == library ✓")

## 7. Naive vs. OOF for one row

The cell below takes one row and reconstructs *which target values* entered its
encoding under each scheme — the leak made visible:

In [ ]:
row = 0  # berlin, y = 1
city = df.loc[row, "city"]
same_city = df.index[df["city"] == city]

naive_sources = same_city                                   # all rows of the city
oof_sources = [i for i in same_city
               if fold_of_row[i] != fold_of_row[row]]       # other folds only

print(f"row {row}: city={city}, y={df.loc[row, 'y']}, fold={fold_of_row[row]}")
print(f"  naive encoding {overview.loc[row, 'city_te_naive']:.3f} "
      f"built from rows {list(naive_sources)} -> targets {list(df.loc[naive_sources, 'y'])}")
print(f"  OOF   encoding {overview.loc[row, 'city_te_oof']:.3f} "
      f"built from rows {oof_sources} -> targets {list(df.loc[oof_sources, 'y'])}")
print(f"  own row {row} in naive sources: {row in list(naive_sources)}")
print(f"  own row {row} in OOF sources:   {row in oof_sources}")

## 8. What remains: the second-order channel

OOF encoding guarantees a row's own target is not in its own feature. It does **not**
guarantee the *training rows'* features are free of the *validation rows'* targets.

Suppose `train_cv` now evaluates fold 0 with these same folds. The model trains on the
rows of folds 1 and 2 — but each of those rows was OOF-encoded using "every fold except
its own", and for all of them that includes fold 0:

In [ ]:
val_fold = 0
train_rows = folds[val_fold][0]

print(f"evaluating fold {val_fold}: validation rows = {[int(i) for i in folds[val_fold][1]]}\n")
r = train_rows[0]
sources = [i for i in df.index
           if df.loc[i, "city"] == df.loc[r, "city"] and fold_of_row[i] != fold_of_row[r]]
leaked = [i for i in sources if fold_of_row[i] == val_fold]
print(f"training row {r} (city={df.loc[r, 'city']}, fold={fold_of_row[r]}):")
print(f"  its OOF encoding was built from rows {sources}")
print(f"  of these, rows {leaked} belong to fold {val_fold} -- the fold being evaluated")

So the chain *fold-0 targets → training rows' features → fitted model → fold-0
predictions* is still open, just heavily diluted (a couple of targets inside a smoothed
mean over many). This is the **second-order optimism** of walkthrough section 6.5:
≈0.52 AUC on pure noise instead of 0.50, versus ≈0.67 for the naive scheme.

Closing it would require nested CV — re-running section 5 *inside* each training set
with inner folds. The remaining sections make this concrete — but first we need to be
precise about a word we have used loosely: *the model*.

## 9. Wait — what "model"? The two learners

Everything up to here was **only the encoder**. No model has been trained in this
notebook yet. A full pipeline has *two* separate learning steps, and both of them look
at the target — which is exactly why the leakage story is confusing:

| | learns what? | from what? | output |
|---|---|---|---|
| **1. Encoder** (`TargetEncoder`) | one number per city (smoothed mean of `y`) | `city` + `y` | the feature column `city_te` |
| **2. Model** (logistic regression, XGBoost, …) | how to map *features* → prediction | feature table (`city_te`, numerics, …) + `y` | predictions |

The CV score everyone reports is about the **model**: train it on 4 folds' feature
rows, predict the held-out fold, compute AUC. (That is what the library's `train_cv`
does.) The encoder is just feature preparation that happens *before* the model —
but because it consumed `y`, it can smuggle target information into the model's
input table.

Let's actually train a model once, so it stops being abstract — logistic regression
on our OOF-encoded feature, for evaluation fold 0:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

train_idx, val_idx = folds[0]
X_train = oof[["city_te"]].iloc[train_idx]   # encoder output = model input
X_val   = oof[["city_te"]].iloc[val_idx]

model = LogisticRegression().fit(X_train, y[train_idx])   # <-- the model, stage 2
print("learned weight on city_te:", round(float(model.coef_[0][0]), 3))
print("predictions for fold 0:   ", model.predict_proba(X_val)[:, 1].round(3))
print("fold-0 AUC:               ", round(roc_auc_score(y[val_idx], model.predict_proba(X_val)[:, 1]), 3))

(Ignore the AUC value itself — 5 validation rows make it meaningless; the point here
is the *mechanics*: fit on training rows' features, predict held-out rows, score.)

The model learned **one weight**: how much to trust `city_te`. (With more features it
would learn one weight each — how much to trust the encoded city *relative to* the
others.) Hold on to that idea; it is where the schemes below will differ.

Every leakage statement so far can now be restated precisely:

- **First-order:** a validation row's *own* `y` is inside its *own* `X` value — the
  model can read the answer off the input at prediction time.
- **Second-order:** the validation fold's `y` values are inside the *training* rows'
  `X` values — the model's *training* was informed by the answers it will be graded on.

## 10. Three encoding schemes for one evaluation fold

There are three reasonable ways to prepare `city_te` when evaluating fold 0. They
differ **only in what the training rows get** — all three encode the validation rows
identically (from the training folds):

- **A — shared-fold OOF** (the library): every row keeps its global OOF value from
  section 6. Training rows' encodings were computed from "all folds but their own" —
  which includes fold 0.
- **B — fit on the training folds** (what an sklearn `Pipeline` inside
  `cross_val_score` would do): fit the encoder on folds 1+2, apply it to those same
  rows. Each training row's own target is back inside its encoding.
- **C — nested**: like B for the validation rows, but the training rows get an *inner*
  OOF among themselves (folds within the training set). Nobody's encoding contains
  their own target, and fold 0's targets touch nothing.

In [ ]:
tr_idx, va_idx = folds[0]
train_cities = df[["city"]].iloc[tr_idx]
train_y = y[tr_idx]

# A -- shared-fold OOF: training rows keep their global OOF encodings (section 6)
te_A = oof["city_te"].iloc[tr_idx].to_numpy()

# B -- fit on the training folds, apply to those same rows
enc_B = TargetEncoder(["city"], smoothing=2.0).fit(train_cities, train_y)
te_B = enc_B.transform(train_cities)["city_te"].to_numpy()

# C -- nested: inner OOF *within* the training folds
inner_folds = make_folds(train_y, task_type="binary", n_splits=2, seed=0)
oof_inner, enc_C = oof_target_encode(train_cities.reset_index(drop=True), train_y,
                                     ["city"], inner_folds, smoothing=2.0)
te_C = oof_inner["city_te"].to_numpy()

pd.DataFrame({
    "city": train_cities["city"].to_numpy(), "y": train_y,
    "A_shared_oof": te_A, "B_fit_on_train": te_B, "C_nested": te_C,
}, index=tr_idx)

Same 10 training rows, three different feature columns. Note that under B, all rows
of a city share one value (like naive encoding — because within the training set, it
*is* naive encoding), while A and C vary within a city.

The validation rows, by contrast, are encoded identically under all three schemes —
always from the training folds only:

In [ ]:
val_cities = df[["city"]].iloc[va_idx]
val_A = oof["city_te"].iloc[va_idx].to_numpy()
val_B = enc_B.transform(val_cities)["city_te"].to_numpy()
val_C = enc_C.transform(val_cities)["city_te"].to_numpy()  # C's outer encoder = fit on all 10 training rows
assert np.allclose(val_A, val_B) and np.allclose(val_B, val_C)
print("validation-row encodings identical under A, B and C \u2713")

So the CV *measurement setup* is equally honest-looking in all three — the difference
is what the model **trained on**. Trace the sources for one training row:

In [ ]:
r = int(tr_idx[0])  # one training row
r_city, r_fold = df.loc[r, "city"], fold_of_row[r]
same_city_train = [int(i) for i in tr_idx if df.loc[i, "city"] == r_city]

inner_fold_of = np.empty(len(tr_idx), dtype=int)
for k, (_, iva) in enumerate(inner_folds):
    inner_fold_of[iva] = k
pos = {int(row): p for p, row in enumerate(tr_idx)}

src_A = [int(i) for i in df.index[df["city"] == r_city] if fold_of_row[i] != r_fold]
src_B = same_city_train
src_C = [i for i in same_city_train if inner_fold_of[pos[i]] != inner_fold_of[pos[r]]]

print(f"training row {r} (city={r_city}), evaluating fold 0:\n")
for name, src in [("A", src_A), ("B", src_B), ("C", src_C)]:
    print(f"  scheme {name}: encoded from rows {src}"
          f"  | own row inside: {r in src}"
          f"  | fold-0 rows inside: {[i for i in src if fold_of_row[i] == 0]}")

There it is, row by row:

- **A** — own row excluded ✓, but fold-0 rows *are* among the sources → second-order.
- **B** — no fold-0 rows ✓, but the row's *own target* is a source → first-order,
  inside the training set.
- **C** — neither. Clean, at the price of the inner-fold machinery. (If C's source
  list prints empty: with only 15 rows, this city has no rows in the *other* inner
  fold, so the encoder falls back to the global mean — still leak-free, just
  uninformative. A tiny-data artifact, not a flaw in the scheme.)

## 11. Why any of this matters: put a model on top

The mechanical difference is clear; does it change outcomes? Two experiments, each
isolating one defect. (These use bigger synthetic data — the 15-row set is too small
to train anything on.)

**Experiment 1 — scheme B damages the model.** Data with *real* signal in a numeric
feature, plus a *pure-noise* high-cardinality categorical. A good pipeline should
learn to use `x_num` and ignore `city_te`. Under B, the training rows' `city_te`
partly *is* the label, so the model over-trusts it — remember the "one weight per
feature" from section 9 — and that misplaced trust costs real AUC on validation.

First, a helper that produces the three encodings per outer fold:

In [ ]:
def scheme_encodings(scheme, df_c, y_arr, outer_folds, smoothing, seed):
    """Yield (tr, va, te_train, te_val) per outer fold under scheme A, B or C."""
    if scheme == "A":
        oof_all, _ = oof_target_encode(df_c, y_arr, ["city"], outer_folds, smoothing)
    for tr, va in outer_folds:
        if scheme == "A":
            te = oof_all["city_te"].to_numpy()
            yield tr, va, te[tr], te[va]
        elif scheme == "B":
            enc = TargetEncoder(["city"], smoothing).fit(df_c.iloc[tr], y_arr[tr])
            yield (tr, va,
                   enc.transform(df_c.iloc[tr])["city_te"].to_numpy(),
                   enc.transform(df_c.iloc[va])["city_te"].to_numpy())
        elif scheme == "C":
            inner = make_folds(y_arr[tr], "binary", n_splits=4, seed=seed)
            oof_in, enc = oof_target_encode(df_c.iloc[tr].reset_index(drop=True),
                                            y_arr[tr], ["city"], inner, smoothing)
            yield (tr, va,
                   oof_in["city_te"].to_numpy(),
                   enc.transform(df_c.iloc[va])["city_te"].to_numpy())

In [ ]:
def cv_signal(scheme, seed):
    """Signal in x_num, pure-noise categorical. Returns (mean AUC, mean |weight| on city_te)."""
    rng = np.random.default_rng(seed)
    n = 3000
    x_num = rng.normal(size=n)                                   # real signal
    y_sig = rng.binomial(1, 1 / (1 + np.exp(-1.5 * x_num)))
    df_c = pd.DataFrame({"city": rng.integers(0, 400, size=n).astype(str)})  # pure noise
    outer = make_folds(y_sig, "binary", n_splits=5, seed=seed)
    aucs, te_weights = [], []
    for tr, va, te_tr, te_va in scheme_encodings(scheme, df_c, y_sig, outer, 10.0, seed):
        Xtr = np.column_stack([x_num[tr], te_tr])
        Xva = np.column_stack([x_num[va], te_va])
        m = LogisticRegression().fit(Xtr, y_sig[tr])
        aucs.append(roc_auc_score(y_sig[va], m.predict_proba(Xva)[:, 1]))
        te_weights.append(abs(m.coef_[0][1]))                    # trust in city_te
    return float(np.mean(aucs)), float(np.mean(te_weights))

for scheme in "ABC":
    res = [cv_signal(scheme, s) for s in range(10)]
    aucs, ws = zip(*res)
    print(f"scheme {scheme}: mean AUC = {np.mean(aucs):.3f}   mean |weight| on noise feature = {np.mean(ws):.2f}")

You should see B roughly **0.02 AUC below** A and C (≈0.79 vs ≈0.81), and its weight
on the pure-noise `city_te` column several times larger. That is the model damage in
one line: B's validation metric is honest, but it honestly reports a *worse model* —
one that learned to trust a feature whose training-time predictiveness was fabricated
by first-order leakage.

**Experiment 2 — scheme A flatters the metric.** Now the walkthrough-6.5 setup: the
target is *pure noise*, the categorical is the only feature, so any AUC above 0.5 is
fabricated. One subtlety: a logistic regression on a single feature can barely exploit
the second-order leak (its AUC depends only on how the feature *ranks* the rows), so
we use a boosted tree — a flexible model that can. Takes ~a minute:

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def cv_noise(scheme, seed):
    """Pure-noise target, high-cardinality categorical as the only feature."""
    rng = np.random.default_rng(seed)
    n = 2000
    y_n = rng.binomial(1, 0.5, size=n)                                        # noise
    df_c = pd.DataFrame({"city": rng.integers(0, 700, size=n).astype(str)})
    outer = make_folds(y_n, "binary", n_splits=5, seed=seed)
    aucs = []
    for tr, va, te_tr, te_va in scheme_encodings(scheme, df_c, y_n, outer, 2.0, seed):
        m = HistGradientBoostingClassifier(max_iter=100, max_depth=4, random_state=0)
        m.fit(te_tr.reshape(-1, 1), y_n[tr])
        aucs.append(roc_auc_score(y_n[va], m.predict_proba(te_va.reshape(-1, 1))[:, 1]))
    return float(np.mean(aucs))

for scheme in "ABC":
    scores = [cv_noise(scheme, s) for s in range(10)]
    print(f"scheme {scheme}: mean AUC on pure noise = {np.mean(scores):.3f}")

You should see A around **0.52** while B and C sit at **0.50**: A's model trained on
features that contained the validation folds' targets, and a flexible model converts
that into fake skill. B and C never let validation targets near the training features,
so on noise they measure exactly chance — B's *metric* is fine; it was its *model*
that suffered in experiment 1.

## 12. Summary

| Scheme | own target in own feature | val targets in train features | model quality | metric honesty | compute |
|---|---|---|---|---|---|
| naive (fit on everything) | yes | yes | damaged | inflated (≈0.67 on noise) | 1× |
| **B** — fit on training folds (sklearn-`Pipeline` style) | yes, within train | no | **damaged** (exp. 1) | honest | ~1× |
| **A** — shared-fold OOF (**the library**) | no | yes | intact | slightly inflated (≈0.52 on noise, exp. 2) | 1× |
| **C** — nested CV | no | no | intact | honest | ~5× |

Every scheme is a trade between three currencies: model quality, metric honesty, and
compute. The library buys intact models and cheap compute with a small, quantified
metric optimism — visible only in pathological setups like experiment 2. When a
decision truly hinges on the last 0.01 of AUC honesty, scheme C (nested CV) is the
escape hatch — it is scheme B for the validation rows plus an inner OOF for the
training rows, i.e. exactly this notebook's section 5 run again *inside* each
training set.